## 0. (선택) 노트북 위치 확인

`DATA_DIR` 상대경로가 맞는지 헷갈리면 아래 셀을 먼저 실행. 두 줄 중 어느 쪽이
`True`가 나오는지 보고, 1번 셀의 `DATA_DIR`을 그에 맞게 `"data/lg/aircon"` 또는
`"../data/lg/aircon"`으로 정한다.

In [ ]:
# 노트북 위치 확인용 - DATA_DIR 상대경로가 맞는지 확실치 않으면 이걸 먼저 돌려보기
from pathlib import Path

print("현재 작업 디렉토리:", Path.cwd())
print("이 안에 보이는 것들:", [p.name for p in Path.cwd().iterdir()])
print()
print("data/lg/aircon 이 존재?:", (Path.cwd() / "data/lg/aircon").exists())
print("../../data/lg/aircon 이 존재?:", (Path.cwd() / "../../data/lg/aircon").exists())

# LG 에어컨 매뉴얼 RAG - 청킹 & 검색 테스트

클래스 없이 **함수 + 딕셔너리**로만 짰습니다. 청크 하나는 그냥 이런 딕셔너리예요:

```python
{"doc_id": "AC_FQ25GN9BKN", "section": "...", "subsection": "...", "body": "...", "id": "...", "text": "..."}
```

셀을 위에서부터 순서대로 실행하면 됩니다. 파싱(1~2번)은 한 번만 돌리고,
검색 부분(4~5번)만 계속 재실행하면서 실험하면 편해요.

In [10]:
import hashlib
import json
import os
import re
from collections import Counter
from pathlib import Path

import pymupdf  # pip install pymupdf
from dotenv import load_dotenv
from langchain_core.documents import Document

load_dotenv(Path.cwd().resolve().parents[1] / ".env")  # OPENAI_API_KEY (질문 생성·정식 채점에 사용). 노트북이 experiments/siyeon/ 안에 있다는 전제

DATA_DIR = Path("../../data/lg/aircon")  # 본인 데이터 폴더 경로로 수정
ERROR_JSON = Path("../../data/lg_aircon_errors.json")  # LG 고객지원 에러코드(CH04, CH05 …) - 매뉴얼 PDF에는 없음

# 에러코드 데이터는 특정 모델 매뉴얼이 아니라 LG 에어컨 공통이라 별도 doc_id로 넣고,
# 모델 필터를 걸 때도 항상 함께 검색되게 한다 (_as_filter 참고).
COMMON_DOC_ID = "LG-AC-COMMON"

# 최신 세대 LG 매뉴얼 공통 대챕터 (세탁기/냉장고/에어컨 모두 확인함)
# 벽걸이형(SQ)은 "리모컨으로 사용하기" 대신 "사용하기"라는 이름을 쓴다.
TOP_LEVEL_SECTIONS = [
    "안전을 위해 주의하기",
    "LG ThinQ 사용하기",
    "알아보기",
    "리모컨으로 사용하기",
    "사용하기",
    "조작부로 사용하기",
    "관리하기",
    "고장 신고 전 확인하기",
    "제품 보증서 보기",
    "부록",
]

# 절차형(1, 2, 3 단계) 사용법 챕터. 목차에 없는 4단계 소제목(AI바람, 소프트바람 등)을 추가 감지한다.
HOWTO_SECTIONS = {"리모컨으로 사용하기", "사용하기", "조작부로 사용하기"}

# 사용법/고장 해결과 무관해서 색인에서 빼는 목차 그룹(L2)
SKIP_GROUPS = {"오픈소스 정보", "폐가전제품 처리 절차", "생활 속 전기안전 캠페인"}

MAX_CHUNK_CHARS = 1200
CHUNK_OVERLAP_CHARS = 100


## 1. 청크 만들기 (헬퍼 함수)

청크는 딕셔너리 하나. `make_chunk()`가 브레드크럼(`[모델명] > 섹션 > 소제목`)이 붙은
`text`와, 벡터DB용 고유 `id`를 미리 계산해서 넣어준다.

In [11]:
def make_chunk(doc_id: str, section: str, subsection: str, body: str, tag: str = "") -> dict:
    """청크 dict 하나를 만든다.

    - text: 임베딩에 들어가는 문자열. 섹션 경로를 앞에 붙여 "어느 맥락의 내용인지"가
      벡터에 같이 들어가게 한다. doc_id는 한 문서 안에서 모든 청크가 공유하는 값이라
      임베딩에는 넣지 않고 메타데이터로만 둔다 (모델 필터는 메타데이터로 건다).
    - id: body의 md5. 파이썬 내장 hash()는 프로세스마다 시드가 달라서 커널을 재시작하면
      같은 청크의 id가 바뀌기 때문에 쓰지 않는다 (결과 JSON 비교가 안 됨).
    """
    breadcrumb = f"{section} > {subsection}" if subsection else section
    text = f"{breadcrumb}\n{body}"
    digest = hashlib.md5(body.encode("utf-8")).hexdigest()[:8]
    chunk_id = re.sub(r"\s+", "_", f"{doc_id}_{section}_{subsection}_{digest}")
    return {
        "doc_id": doc_id,
        "section": section,
        "subsection": subsection,
        "body": body,
        "tag": tag,
        "id": chunk_id,
        "text": text,
    }


def split_oversized(chunks: list[dict]) -> list[dict]:
    """MAX_CHUNK_CHARS를 넘는 청크는 줄 경계에서 자르고, 앞 조각 끝을 CHUNK_OVERLAP_CHARS만큼
    다음 조각 앞에 겹쳐 준다 (문장 중간에서 잘려 뜻이 끊기는 것을 줄이기 위해)."""
    result = []
    for c in chunks:
        if len(c["body"]) <= MAX_CHUNK_CHARS:
            result.append(c)
            continue
        lines = c["body"].split("\n")
        piece: list[str] = []
        size = 0
        for line in lines:
            if size + len(line) + 1 > MAX_CHUNK_CHARS and piece:
                result.append(make_chunk(c["doc_id"], c["section"], c["subsection"], "\n".join(piece), c["tag"]))
                # 오버랩: 앞 조각의 마지막 줄들을 CHUNK_OVERLAP_CHARS 안에서 가져온다
                carry, carry_size = [], 0
                for prev in reversed(piece):
                    if carry_size + len(prev) > CHUNK_OVERLAP_CHARS:
                        break
                    carry.insert(0, prev)
                    carry_size += len(prev) + 1
                piece, size = carry, carry_size
            piece.append(line)
            size += len(line) + 1
        if piece:
            result.append(make_chunk(c["doc_id"], c["section"], c["subsection"], "\n".join(piece), c["tag"]))
    return result


def to_document(c: dict) -> Document:
    """청크 dict -> LangChain Document. page_content(임베딩 대상)는 text, 나머지는 메타데이터."""
    return Document(
        page_content=c["text"],
        metadata={
            "id": c["id"], "doc_id": c["doc_id"], "section": c["section"],
            "subsection": c["subsection"], "body": c["body"], "tag": c["tag"],
        },
    )


## 2. PDF 구조 분석 함수들

- `locate_sections` — 대챕터(안전/ThinQ/알아보기/사용하기/관리하기/고장신고/보증서/부록) 페이지 범위
- `toc_headings` — PDF 북마크에서 2·3단계 소제목 (10개 모델 전부 본문 줄과 100% 일치 확인)
- `parse_troubleshooting` — 고장신고 표 → **증상 1개 = 청크 1개**
- `parse_by_toc` — 나머지 전부: 목차 2·3단계 경계로 분리, 사용법 챕터는 4단계 숨은 소제목까지

자세한 배경은 `experiments/siyeon/docs/experiment_notes.md` 참고.


In [12]:
def extract_page_texts(doc) -> list[str]:
    """페이지별 원문. 러닝헤더/페이지 번호를 그대로 둔다 (locate_sections가 그 패턴을 쓰기 때문).
    본문으로 쓸 때는 strip_page_furniture()를 거친다."""
    return [page.get_text() for page in doc]


def strip_page_furniture(text: str) -> str:
    """페이지 앞머리의 페이지 번호와 러닝헤더(챕터명)를 떼어낸다.

    페이지 텍스트는 항상  "38\\n관리하기\\n(본문…)"  또는 챕터 시작 페이지면
    "36\\n관리하기\\n관리하기\\n청소하기…" 처럼 시작한다. 이걸 안 떼면 청크마다
    "38 관리하기" 같은 잡음이 본문 중간에 박혀 임베딩을 흐린다.
    """
    lines = text.split("\n")
    k = 0
    while k < len(lines) and k < 4:
        s = lines[k].strip()
        if s == "" or s.isdigit() or s in TOP_LEVEL_SECTIONS:
            k += 1
        else:
            break
    return "\n".join(lines[k:])


def locate_sections(page_texts: list[str]) -> dict[str, tuple[int, int]]:
    """각 대챕터가 몇 페이지(인덱스)에서 시작하는지 찾는다.

    챕터가 '진짜로 시작하는' 페이지는 프레임메이커 특유의 레이아웃 때문에
    제목이 인접한 두 줄에서 연속으로 반복된다 (러닝헤더 + 본문 제목):
        46
        고장 신고 전 확인하기      <- 러닝헤더
        고장 신고 전 확인하기      <- 진짜 챕터 제목

    '첫 3줄 안에 제목이 있으면 챕터 시작'으로 판단하면, 목차 페이지에서
    줄바꿈이 우연히 겹칠 때 목차 페이지 자체를 챕터 시작으로 오인한다
    (실제로 AC_FQ18GC1EHN에서 이 버그로 '고장 신고 전 확인하기' 챕터가
    목차 페이지 단 1장으로 쪼그라들어 카테고리가 0개로 나온 적 있음).
    그래서 '두 줄 연속 반복'을 진짜 챕터 시작의 조건으로 삼는다.
    """
    starts: dict[str, int] = {}
    for i, text in enumerate(page_texts):
        lines = [ln.strip() for ln in text.strip().split("\n")]
        for section in TOP_LEVEL_SECTIONS:
            if section in starts:
                continue
            for j in range(min(6, len(lines) - 1)):  # 러닝헤더는 페이지 앞부분에 있음
                if lines[j] == section and lines[j + 1] == section:
                    starts[section] = i
                    break

    ordered = sorted(starts.items(), key=lambda kv: kv[1])
    ranges = {}
    for idx, (name, start) in enumerate(ordered):
        end = ordered[idx + 1][1] if idx + 1 < len(ordered) else len(page_texts)
        ranges[name] = (start, end)
    return ranges


def toc_headings(doc) -> list[dict]:
    """PDF에 내장된 목차(북마크)에서 2·3단계 제목을 가져온다.

    LG 매뉴얼은 프레임메이커로 만들어져 목차가 PDF 북마크로 그대로 들어있고,
    10개 모델 전부에서 2·3단계 제목이 본문 줄과 100% 글자 그대로 일치하는 걸 확인했다.
    1단계(대챕터)는 파일마다 중첩이 어긋나는 경우가 있어서 쓰지 않고 locate_sections에 맡긴다.
    """
    return [
        {"level": level, "title": title.strip(), "page_idx": page_no - 1}
        for level, title, page_no in doc.get_toc()
        if level in (2, 3)
    ]


In [13]:
# ── 고장 신고 전 확인하기 > 문제 해결하기: 표(table) 기반 추출 ──
# find_tables()로 [증상, 원인 및 해결책] 표를 그대로 읽는다.
# 카테고리(운전/소음/와이파이/공기청정/클린봇)는 모델 등급마다 개수가 달라서
# 하드코딩하지 않고 페이지에서 그때그때 감지한다.
#
# 청크 단위는 "증상 1개" — 한 증상에 딸린 원인·해결책 여러 개를 전부 한 청크에 담는다.
# (원인 1행 = 청크 1개로 쪼갰을 때는 '와이파이 연결 안 돼요' 같은 질문에 같은 증상의
#  원인 행 6개가 top_k를 독식해서 답변이 반토막 나고, 100~300자짜리 행이 전체 청크의
#  절반을 차지해 다른 섹션 검색까지 방해했다.)

KNOWN_CATEGORY_WORDS = {"운전", "소음", "와이파이", "공기청정", "클린봇", "제습", "냄새"}


def find_categories_on_page(page) -> list[tuple[str, float]]:
    """페이지에서 카테고리 소제목을 (이름, y좌표) 형태로 찾는다."""
    found = []
    for block in page.get_text("blocks"):  # (x0, y0, x1, y1, text, block_no, block_type)
        text = block[4].strip()
        if text.replace("(옵션)", "") in KNOWN_CATEGORY_WORDS:
            found.append((text, block[1]))
    return found


def category_for_table(table, categories_on_page: list[tuple[str, float]], fallback: str) -> str:
    """표 바로 위(y좌표가 표보다 작고, 가장 가까운)에 있는 카테고리 이름을 찾는다."""
    if not categories_on_page:
        return fallback
    table_top_y = table.bbox[1]
    candidates = [(name, y) for name, y in categories_on_page if y < table_top_y]
    if not candidates:
        # 표 위에 카테고리 제목이 없으면 앞 페이지에서 이어진 표 → 직전 카테고리 유지.
        # (페이지 아래쪽의 다음 카테고리 제목을 잘못 가져오면 '운전' 표 끝부분이 '소음'으로 붙는다)
        return fallback
    return max(candidates, key=lambda item: item[1])[0]


def _tidy_cell(cell: str | None) -> str:
    """표 셀 안의 줄바꿈(레이아웃용)을 공백으로 펴고, 불릿(•) 앞에서만 줄을 나눈다."""
    if not cell:
        return ""
    flat = re.sub(r"\s*\n\s*", " ", cell.strip())
    return re.sub(r"\s*•\s*", "\n• ", flat).strip()


def parse_troubleshooting(doc, doc_id: str, start: int, end: int) -> list[dict]:
    chunks = []
    current_category = "운전"
    current: dict | None = None  # {"symptom", "category", "parts": [...]}

    def flush():
        if current and current["parts"]:
            body = f"증상: {current['symptom']}\n" + "\n".join(current["parts"])
            chunks.append(make_chunk(doc_id, "고장 신고 전 확인하기", current["category"], body, tag="symptom"))

    for i in range(start, end):
        page = doc[i]
        categories_on_page = find_categories_on_page(page)

        for table in page.find_tables().tables:
            rows = table.extract()
            if not rows or rows[0][:2] != ["증상", "원인 및 해결책"]:
                continue  # 문제해결 표가 아니면 스킵 (보증서 표 등)

            category = category_for_table(table, categories_on_page, current_category)
            current_category = category

            for row in rows[1:]:
                symptom = re.sub(r"\s+", " ", (row[0] or "")).strip()
                cause_solution = _tidy_cell(row[1])
                if not cause_solution:
                    continue
                # 증상 칸이 비어 있으면 같은 증상의 추가 원인. 증상이 페이지를 넘어가면
                # 다음 페이지 표에 같은 증상명이 다시 찍히므로 그 경우도 이어붙인다.
                if symptom and not (current and current["symptom"] == symptom):
                    flush()
                    current = {"symptom": symptom, "category": category, "parts": []}
                if current is None:  # 표가 증상 없이 시작하는 비정상 케이스 방어
                    current = {"symptom": "(증상 미상)", "category": category, "parts": []}
                current["parts"].append(cause_solution)
    flush()
    return chunks


In [14]:
# ── 목차(TOC) 기반 소제목 분리: 안전주의 / ThinQ / 알아보기 / 사용하기 / 관리하기 / 보증서 / 부록 공용 ──
# 2단계(그룹) > 3단계(소제목) 경계에서 청크를 끊는다. 예)
#   관리하기 > 청소하기 > 필터 청소하기          ← 청소 주기 표까지 한 청크
#   관리하기 > 냄새 제거하기 > 냄새에 따른 제거 방법 알아보기
#   안전을 위해 주의하기 > 경고 > 제품을 설치할 때  (경고/주의에 같은 소제목이 반복되므로 그룹까지 붙임)
#
# 사용법(HOWTO) 챕터는 목차에 없는 4단계 소제목(AI 바람, 아이스 쿨파워 냉방 …)이 있어서,
# "짧은 제목 줄 뒤 몇 줄 안에 숫자 단독 줄('1')이 온다" 패턴으로 한 단계 더 쪼갠다.

def looks_like_hidden_header(stripped: str, ahead: list[str]) -> bool:
    return (
        0 < len(stripped) <= 20
        and not stripped[0].isdigit()
        and not stripped.startswith(("•", "-", "~", "*"))
        and not stripped.endswith(("요.", "다.", "니다", "세요", ":", "?"))
        and "1" in ahead
    )


def parse_by_toc(page_texts, toc, doc_id, start, end, section,
                 detect_hidden=False, skip_groups=frozenset(), tag="generic", lookahead=4) -> list[dict]:
    chunks = []
    buf: list[str] = []
    group: str | None = None      # 2단계
    heading: str | None = None    # 3단계
    hidden: str | None = None     # 4단계 (사용법 챕터만)

    def flush():
        body = "\n".join(buf).strip()
        if len(body) > 30 and group not in skip_groups:
            label = " > ".join(x for x in (group, heading, hidden) if x)
            chunks.append(make_chunk(doc_id, section, label, body, tag))

    for i in range(start, end):
        # 이 페이지(또는 바로 다음 페이지)에 시작한다고 목차에 적힌 제목만 경계 후보로 본다
        # → 본문 안 상호참조("~ 관리하기(필터 청소하기)를 보세요")나 다른 페이지의 같은 단어에 안 걸림
        titles_here = {h["title"]: h["level"] for h in toc if h["page_idx"] in (i, i - 1)}
        lines = strip_page_furniture(page_texts[i]).split("\n")
        for j, line in enumerate(lines):
            stripped = line.strip()
            level = titles_here.get(stripped)
            if level == 2 and stripped == group:
                # 그룹 제목이 본문 안에서 소제목으로 한 번 더 쓰인 경우
                # (예: '공기청정 기능 사용하기' 그룹 안의 '공기청정 기능 사용하기' 절차) → 4단계로 취급
                level = None
                if detect_hidden:
                    flush(); buf = []
                    hidden = stripped
                    continue
            if level == 2:
                flush(); buf = []
                group, heading, hidden = stripped, None, None
                continue
            if level == 3:
                flush(); buf = []
                heading, hidden = stripped, None
                continue
            if detect_hidden:
                ahead = [lines[k].strip() for k in range(j + 1, min(j + 1 + lookahead, len(lines)))]
                if looks_like_hidden_header(stripped, ahead):
                    flush(); buf = []
                    hidden = stripped
                    continue
            buf.append(line)
    flush()
    return split_oversized(chunks)


In [15]:
def parse_pdf(pdf_path: Path) -> list[dict]:
    """PDF 한 개 -> 청크 리스트. 섹션마다 위에서 정의한 함수를 다르게 호출한다."""
    doc_id = pdf_path.stem
    doc = pymupdf.open(str(pdf_path))
    page_texts = extract_page_texts(doc)
    toc = toc_headings(doc)
    section_ranges = locate_sections(page_texts)

    chunks: list[dict] = []
    for section, (start, end) in section_ranges.items():
        if section == "고장 신고 전 확인하기":
            # 표 부분('문제 해결하기')은 표 추출로, 그 앞의 '고장 진단하기'(ThinQ 스마트진단 등)는 텍스트로
            chunks += parse_troubleshooting(doc, doc_id, start, end)
            chunks += parse_by_toc(page_texts, toc, doc_id, start, end, section,
                                   skip_groups={"문제 해결하기"}, tag="diagnosis")
        elif section in HOWTO_SECTIONS:
            chunks += parse_by_toc(page_texts, toc, doc_id, start, end, section, detect_hidden=True, tag="howto")
        elif section == "안전을 위해 주의하기":
            chunks += parse_by_toc(page_texts, toc, doc_id, start, end, section, tag="safety")
        else:
            chunks += parse_by_toc(page_texts, toc, doc_id, start, end, section, skip_groups=SKIP_GROUPS)
    doc.close()
    return chunks


def load_error_code_chunks(json_path: Path = ERROR_JSON) -> list[dict]:
    """LG 고객지원 에러코드 JSON -> 청크. 이미 항목(title/content) 단위로 나뉘어 있어 1항목 = 1청크.

    매뉴얼 PDF에는 CH04/CH05 같은 에러코드가 전혀 없다 (증상 기반 문제해결 표만 있음).
    "CH05 떠요" 류 질문은 이 데이터가 없으면 답할 수 없다.
    """
    if not json_path.exists():
        print(f"[경고] 에러코드 JSON 없음: {json_path} - 에러코드 청크 없이 진행")
        return []
    chunks = []
    for product in json.loads(json_path.read_text(encoding="utf-8")):
        for sec in product["sections"]:
            body = sec["content"].strip()
            chunks.append(make_chunk(COMMON_DOC_ID, "에러코드", sec["title"].strip(), body, tag="error_code"))
    return split_oversized(chunks)


def build_all_chunks(pdf_dir: Path = DATA_DIR) -> list[dict]:
    chunks = []
    for pdf_path in sorted(pdf_dir.glob("*.pdf")):
        chunks += parse_pdf(pdf_path)
    chunks += load_error_code_chunks()
    return chunks


## 3. 파싱 실행 & 확인

여기서부터는 PDF들을 실제로 파싱한다. 이 셀은 한 번만 돌리면 되고,
청크 개수/섹션 분포가 이상하면 위쪽 파싱 함수만 고쳐서 이 셀부터 다시 실행하면 된다.

In [16]:
chunks = build_all_chunks(DATA_DIR)
print(f"총 {len(chunks)}개 청크")
print("섹션별 개수:", Counter(c["section"] for c in chunks))
print("태그별 개수:", Counter(c["tag"] for c in chunks))
lens = sorted(len(c["body"]) for c in chunks)
print(f"body 길이: min={lens[0]} / 중간값={lens[len(lens)//2]} / max={lens[-1]}")


총 988개 청크
섹션별 개수: Counter({'고장 신고 전 확인하기': 236, '리모컨으로 사용하기': 184, '안전을 위해 주의하기': 140, '관리하기': 114, '사용하기': 106, '알아보기': 59, 'LG ThinQ 사용하기': 54, '제품 보증서 보기': 40, '조작부로 사용하기': 24, '부록': 20, '에러코드': 11})
태그별 개수: Counter({'howto': 314, 'generic': 287, 'symptom': 211, 'safety': 140, 'diagnosis': 25, 'error_code': 11})
body 길이: min=37 / 중간값=324 / max=1199


In [18]:
# 문제해결 카테고리가 모델마다 몇 개씩 잡히는지 확인 (3~5개가 정상)
# 공통 문서(LG-AC-COMMON)는 매뉴얼이 아니라 에러코드 JSON이므로 증상 청크가 없다 → 에러코드 개수만 표시
for doc_id in sorted({c["doc_id"] for c in chunks}):
    if doc_id == COMMON_DOC_ID:
        n_err = sum(1 for c in chunks if c["doc_id"] == doc_id and c["tag"] == "error_code")
        print(doc_id, "-> 에러코드", n_err, "개 (공통 문서, 증상 표 없음)")
        continue
    cats = Counter(c["subsection"] for c in chunks if c["doc_id"] == doc_id and c["tag"] == "symptom")
    print(doc_id, "->", dict(cats))

# 특정 모델의 청크 목록을 눈으로 훑어보기 (섹션 경로가 제대로 잡혔는지)
def show_chunks(doc_id: str, section: str | None = None) -> None:
    for c in chunks:
        if c["doc_id"] == doc_id and (section is None or c["section"] == section):
            preview = c["body"].replace("\n", " ")[:70]
            print(f'{len(c["body"]):5d} {c["tag"]:9s} {c["section"]} > {c["subsection"]}  ::  {preview}')

# show_chunks("AC_FQ18GC1EHN", "관리하기")


AC_FQ16FV6EDN -> {'운전': 15, '소음': 4, '와이파이': 1}
AC_FQ17FN5BDN -> {'운전': 16, '소음': 4, '공기청정(옵션)': 4, '클린봇(옵션)': 2, '와이파이': 1}
AC_FQ18GC1EHN -> {'운전': 15, '소음': 4, '와이파이': 1}
AC_FQ18GN7BKN -> {'운전': 15, '소음': 4, '공기청정(옵션)': 4, '클린봇(옵션)': 2, '와이파이': 1}
AC_FQ18GU1BHN -> {'운전': 15, '소음': 4, '와이파이': 1}
AC_FQ25GN9BKN -> {'운전': 15, '소음': 4, '공기청정(옵션)': 4, '클린봇(옵션)': 2, '와이파이': 1}
AC_SQ06EJ1WAJ -> {'운전': 13, '소음': 4, '와이파이': 1}
AC_SQ07GA3WBN -> {'운전': 13, '소음': 4, '와이파이': 1}
AC_SQ07GJ1WEN -> {'운전': 13, '소음': 4, '와이파이': 1}
AC_SQ09GK1WEN -> {'운전': 13, '소음': 4, '와이파이': 1}
LG-AC-COMMON -> 에러코드 11 개 (공통 문서, 증상 표 없음)


## 4. 임베딩 + 검색 (LangChain: Document + Chroma)

청크 딕셔너리를 LangChain `Document`로 감싸고, `Chroma`에 넣어서 검색한다.
임베딩 모델은 그대로 `BAAI/bge-m3`(`HuggingFaceEmbeddings`로 로드, 최초 1회 ~2GB 다운로드).

`similarity_search_with_score`가 돌려주는 score는 **거리(distance, 낮을수록 유사)**
라서, 팀 평가 모듈(`pipeline/evaluate.py`)의 `distance` 필드 정의와 그대로 맞아떨어진다
(이전 numpy 버전은 반대로 '높을수록 유사'한 코사인 유사도였어서 매번 `1 - score`로
변환해야 했는데, 이제 그럴 필요가 없다).

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

documents = [to_document(c) for c in chunks]

# show_progress_bar는 encode_kwargs 안이 아니라 최상위 show_progress 파라미터로 줘야 함.
# (HuggingFaceEmbeddings가 내부에서 encode(show_progress_bar=self.show_progress, **encode_kwargs)로
#  호출하는데, encode_kwargs 안에 같은 키를 넣으면 중복 인자 TypeError가 남)
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    encode_kwargs={"normalize_embeddings": True},
    show_progress=True,
)
vectorstore = Chroma.from_documents(documents, embeddings, ids=[c["id"] for c in chunks])
print(f"{len(documents)}개 문서 임베딩 완료")

c:\Users\Admin\Desktop\ask-my-appliance\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 31/31 [05:40<00:00, 10.98s/it]


988개 문서 임베딩 완료


In [20]:
# 이미 임베딩된 vectorstore에 에러코드 청크만 추가한다 (전체 재임베딩 불필요, 11개라 몇 초).
# 위 15번 셀을 처음부터 다시 돌린 경우엔 build_all_chunks()가 이미 포함하므로 이 셀은 건너뛰어도 된다.
# 여러 번 실행해도 안전: 기존 에러코드 청크를 지우고 다시 넣는다.
error_chunks = load_error_code_chunks()

if not any(c["doc_id"] == COMMON_DOC_ID for c in chunks):
    chunks += error_chunks

old_ids = vectorstore.get(where={"doc_id": COMMON_DOC_ID})["ids"]
if old_ids:
    vectorstore.delete(ids=old_ids)
vectorstore.add_documents([to_document(c) for c in error_chunks], ids=[c["id"] for c in error_chunks])
print(f"에러코드 청크 {len(error_chunks)}개 추가 -> vectorstore 총 {vectorstore._collection.count()}개")


Batches: 100%|██████████| 1/1 [00:05<00:00,  5.57s/it]

에러코드 청크 11개 추가 -> vectorstore 총 988개


In [21]:
# 사용자가 등록할 수 있는 모델 목록. 에러코드 공통 문서는 여기서 빼고, 필터 시 항상 같이 붙인다.
known_doc_ids = sorted({c["doc_id"] for c in chunks} - {COMMON_DOC_ID})


def _as_filter(doc_id: str | list[str] | None) -> dict | None:
    """doc_id(들)를 Chroma 메타데이터 필터로 바꾼다. None이면 필터 없음(전체 검색).

    존재하지 않는 doc_id는 여기서 바로 막는다 - Chroma는 없는 값으로 필터를 걸면
    에러 없이 빈 결과만 돌려줘서, 오타(FQ18GC1EHN vs AC_FQ18GC1EHN)를
    "검색이 안 된다"로 오해하기 쉽기 때문.
    """
    if doc_id is None:
        return None
    ids = [doc_id] if isinstance(doc_id, str) else list(doc_id)
    unknown = [d for d in ids if d not in known_doc_ids]
    if unknown:
        raise ValueError(f"알 수 없는 모델 {unknown}\n선택 가능: {known_doc_ids}")
    # 에러코드(LG 에어컨 공통)는 어떤 모델을 골라도 같이 검색 대상에 넣는다
    return {"doc_id": {"$in": ids + [COMMON_DOC_ID]}}


def search(query: str, doc_id: str | list[str] | None = None, top_k: int = 3) -> list[tuple[dict, float]]:
    """vectorstore를 감싸는 얇은 래퍼. 최종 웹 서비스의 검색 API와 같은 형태.

    웹에서는 사용자가 가전을 미리 등록해두므로 모델 ID가 질문 문장과 별개로
    들어온다. 그래서 질문에서 모델명을 추출하지 않고 doc_id를 명시적으로 받는다.
      - doc_id=None        : 전체 문서에서 검색 (팀 공용 평가용)
      - doc_id="AC_..."    : 해당 모델 문서 + LG 공통 에러코드
      - doc_id=["AC_...",] : 여러 대 등록한 사용자 시나리오 ($in 필터)

    반환은 (chunk_dict, distance) 튜플 목록. distance는 낮을수록 유사.
    """
    results = vectorstore.similarity_search_with_score(query, k=top_k, filter=_as_filter(doc_id))
    return [({**doc.metadata, "text": doc.page_content}, float(distance)) for doc, distance in results]


def inspect_retrieval(queries: list[str], doc_id: str | list[str] | None = None, top_k: int = 3) -> None:
    """생성 모델 없이 검색 결과만 눈으로 확인. distance는 낮을수록 유사."""
    scope = f"[모델: {doc_id}]" if doc_id else "[전체 문서]"
    for query in queries:
        print(f"\n{'=' * 60}\nQ: {query}  {scope}")
        for rank, (chunk, distance) in enumerate(search(query, doc_id=doc_id, top_k=top_k), start=1):
            print(f"\n  [{rank}] distance={distance:.3f}  {chunk['doc_id']} > {chunk['section']} > {chunk['subsection']}")
            preview = chunk["body"].strip().replace("\n", " ")[:150]
            print(f"      {preview}...")


## 5. 검색 테스트

이 아래 셀만 계속 바꿔가면서 재실행하면 됨 (위쪽 임베딩은 다시 안 돌려도 됨).

`TARGET_MODEL`이 웹에서 "사용자가 등록해둔 가전" 역할. 이 값만 바꿔서 모델별로 테스트한다.
`None`으로 두면 전체 문서에서 검색.


In [22]:
# 웹에서 사용자가 등록해둔 가전을 흉내내는 값. 이것만 바꿔서 모델별 테스트.
TARGET_MODEL = "AC_FQ18GC1EHN"     # None 이면 전체 문서 검색
print("선택 가능한 모델:", known_doc_ids)

test_queries = [
    "에어컨 취침 모드는 어떻게 켜요?",
    "와이파이 연결이 안 돼요",
    "필터는 얼마나 자주 청소해야 하나요?",
    "에어컨에서 이상한 냄새가 나요",
    "CH05 에러가 떴어요",                 # 에러코드 JSON(LG 공통)에서 나와야 함
]

inspect_retrieval(test_queries, doc_id=TARGET_MODEL, top_k=3)


선택 가능한 모델: ['AC_FQ16FV6EDN', 'AC_FQ17FN5BDN', 'AC_FQ18GC1EHN', 'AC_FQ18GN7BKN', 'AC_FQ18GU1BHN', 'AC_FQ25GN9BKN', 'AC_SQ06EJ1WAJ', 'AC_SQ07GA3WBN', 'AC_SQ07GJ1WEN', 'AC_SQ09GK1WEN']

Q: 에어컨 취침 모드는 어떻게 켜요?  [모델: AC_FQ18GC1EHN]


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.21it/s]



  [1] distance=0.651  AC_FQ18GC1EHN > 리모컨으로 사용하기 > 추가 기능 및 설정하기 > AI 수면(취침) 설정하기
      사용자의 숙면을 돕는 AI 수면(취침) 운전을 설정할 수  있습니다. 1 리모컨의 전원 버튼을 눌러서 제품을 켜세요. 2 리모컨의 취침 버튼을 누르세요. 3 취침 버튼을 반복적으로 눌러 원하는 시간을  설정하세요. 예약이 완료됩니다. • 30분부터 시작해 이후 1시간 ...

  [2] distance=0.764  AC_FQ18GC1EHN > 리모컨으로 사용하기 > 리모컨 살펴보기 > 리모컨 처음 사용하기
      바람세기 + - 바람 세기를 설정할 수 있습니다. • 바람 세기는 1 ↔ 2 ↔ 3 ↔ 4 ↔ 5단계까지 설정할 수 있습니다. 버튼 설명  알아두기 • 리모컨을 일정 시간 동안 사용하지 않을 경우 리모컨이 대기모드로 전환되며 리모컨의 표시부에 나타나는 기능이  실내기 ...

  [3] distance=0.794  AC_FQ18GC1EHN > 리모컨으로 사용하기 > 냉방 기본 기능 작동하기 > 냉방 기능 사용하기
      제품의 냉방 기능이 작동하면 좌우 토출구에서 차가운  바람이 나옵니다. 냉방 기능은 가장 기본적인 운전 모드이며 리모컨으로  쉽게 작동할 수 있습니다. 1 리모컨의 전원 버튼을 눌러 제품을 켜세요. 2 리모컨의 운전선택 버튼을 반복적으로 눌러 냉방을  선택하세요. 리모...

Q: 와이파이 연결이 안 돼요  [모델: AC_FQ18GC1EHN]


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.91it/s]



  [1] distance=0.646  AC_FQ18GC1EHN > 고장 신고 전 확인하기 > 와이파이
      증상: 제품과 스마트폰을 와이파이로 연결할 수 없어요. 스마트폰에 연결된 와이파이 비밀번호가 다른가요? • 스마트폰의 바탕 화면에서 설정 버튼을 누른 다음 네트워크 목록에서 사용 중인 와이파이를 찾으세요. 버튼을 길게 눌러 네트워크 지우기를 선택한 다음 처음부터 제품 ...

  [2] distance=0.887  AC_FQ18GC1EHN > LG ThinQ 사용하기 > LG ThinQ와 LG 가전 연결하기 > 앱 설치 및 제품 등록하기
      LG ThinQ 앱을 설치하면 언제 어디서나 편리하게 우리집  LG 가전을 관리할 수 있습니다. • 제품에 와이파이 기능이 있는 모델에만 적용됩니다. QR 코드가 부착된 제품 스마트폰의 카메라 또는 QR 코드 리더 앱을 사용하여  제품에 부착된 QR 코드를 스캔하세요....

  [3] distance=0.912  AC_FQ18GC1EHN > 고장 신고 전 확인하기 > 고장 진단하기
      제품의 고장 원인을 진단할 수 있습니다. 알아두기 • LG전자의 과실이 아닌 외부 요인(와이파이 사용 불가,  와이파이 연결 해제, 앱 스토어 정책 변경, 앱 사용 불가  등)에 의해 서비스가 중지될 수 있습니다. • 사전 고지 없이 변경될 수 있으며, 현지 사정에 따...

Q: 필터는 얼마나 자주 청소해야 하나요?  [모델: AC_FQ18GC1EHN]


...


  [1] distance=0.478  AC_FQ18GC1EHN > 관리하기 > 청소하기 > 필터 청소하기
      일정한 주기마다 제품에서 필터를 분리하여 청소하세요. 필터 청소 주기  팁 • 필터는 LG전자 홈페이지 www.lge.com에서 구입할 수  있습니다. 주의 • 제품에서 필터를 분리하기 전에 제품의 전원을 끈 다음  전원 플러그를 빼거나 주 전원 스위치를 내리세요. •...

  [2] distance=0.667  AC_FQ18GC1EHN > 관리하기 > 청소하기 > 필터 청소하기
      극세 필터 청소하기 1 공기청정 모델은 극세 필터 안에 있는 알러지케어 집진  필터와 극세 필터를 분리하세요. 2 극세 필터에 오염 물질이 적을 때는 흐르는 물에 필터를  씻으세요. 극세 필터에 먼지 같은 오염 물질이 많을  때는 중성 세제를 푼 물에 필터를 담갔다가 ...

  [3] distance=0.768  AC_FQ18GC1EHN > 관리하기 > 청소하기 > 필터 교체 알림 초기화하기(옵션)
      본 내용은 공기청정 기능이 있는 모델에 적용됩니다. 제품이 작동한 시간을 고려하여 필터를 교체하는 시기를  알려줍니다. 필터를 교체해야 하는 시기가 되면 실내기  표시부에 필터 교체 알림 아이콘이 깜빡입니다.  필터를 청소하거나 교체한 후 필터 교체 알림을  초기화하세...

Q: 에어컨에서 이상한 냄새가 나요  [모델: AC_FQ18GC1EHN]


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.46it/s]



  [1] distance=0.667  AC_FQ18GC1EHN > 관리하기 > 냄새 제거하기 > 냄새에 따른 제거 방법 알아보기
      제품에서 이상한 냄새가 날 때는 아래 방법을 참고하여  냄새를 없애세요. 매캐하고 쾨쾨한 냄새 제품을 오랜 시간 사용해서 제품 내부에 수분이 많을 때는  이런 냄새가 날 수 있습니다. • 창문을 열고 실내를 환기하면서 공기청정 또는 송풍  기능으로 1시간 이상 충분히 ...

  [2] distance=0.852  AC_FQ18GC1EHN > 고장 신고 전 확인하기 > 소음
      증상: 실외기의 팬 돌아가는 소리가 계속 바뀌어요. 버튼을 눌러 바람 세기를 바꾸거나 또는 실내 온도가 희망하는 온도가 됐나요? • 실내 온도가 희망하는 온도에 가까워지면 효율적으로 팬의 회전수를 조절하여 절전 운전을 합니다....

  [3] distance=0.860  AC_FQ18GC1EHN > 고장 신고 전 확인하기 > 운전
      증상: 실내기에서 흰 안개가 나와요. 실내의 습도가 높거나 제품을 설치한 곳이 음식점이나 기름을 많이 사용하는 곳인가요? • 실내 습도가 높을 때는 실내기의 좌우 토출구를 통해서 수증기가 나올 수 있습니다. 기름을 많이 사용하는 곳에 제품을 설치했다면 열교환기를 정기적...

Q: CH05 에러가 떴어요  [모델: AC_FQ18GC1EHN]


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.85it/s]


  [1] distance=0.655  LG-AC-COMMON > 에러코드 > CH05 / E0 / CH53 (통신 에러)
      CH05 / E0 / CH53 에러는 실내기와 실외기의 통신 이상으로 발생할 수 있습니다. 제품 신규 또는 이전 설치 직후 발생한다면, 인버터 제품에서 실내기와 실외기의 통신 이상을 감지하는 기능이 작동한 것이며 전문 서비스매니저의 확인이 필요합니다. 신규/이전 설치가...

  [2] distance=0.794  LG-AC-COMMON > 에러코드 > CH04 / FL (배수 불량, 만수 감지)
      CH 04 에러는 배수 불량(실내기 Dain 불량)/만수 감지 경고입니다. [천장형 에어컨] 에어컨을 사용 중 CH04 에러가 뜬다면, 실내기에서 생성된 물이 제대로 배출되지 않아 발생하는 에러입니다. 제품의 전원 코드 또는 전용 차단기를 내렸다가 약 5분 후 다시 올...

  [3] distance=0.802  LG-AC-COMMON > 에러코드 > CH10 / E6 / CH67 / EF (실내기·실외기 팬 구속)
      CH10 / E6 에러는 실내기 팬 구속, CH67 / EF 에러는 실외기 팬 구속에 의한 동작 불량 또는 팬 모터 불량 시 발생하는 에러입니다. 실내기 및 실외기 팬 구동부에 이물질이나 눈이 쌓인 경우 발생할 수 있으니 확인 후 제거해 주세요. 환경적으로 이상이 없다...


### 같은 질문, 모델만 바꿔서 비교

질문 문장은 그대로 두고 `doc_id`만 바꿔서 검색한다 (질문에 모델명을 섞으면 임베딩에
노이즈가 들어가서 실제 서비스와 조건이 달라짐).
클린봇이 **없는** 모델(FQ16)에서 top1으로 뭐가 올라오고 distance가 얼마나 벌어지는지 봐야,
나중에 "이 모델에는 해당 기능이 없습니다" 판단 기준(distance 임계값)을 잡을 수 있다.


In [23]:
query = "클린봇 기능 어떻게 써요?"

for model in ["AC_FQ25GN9BKN", "AC_FQ16FV6EDN"]:   # 상위 모델(클린봇 있음) / 하위 모델(클린봇 없음)
    inspect_retrieval([query], doc_id=model, top_k=3)



Q: 클린봇 기능 어떻게 써요?  [모델: AC_FQ25GN9BKN]


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.48it/s]



  [1] distance=0.532  AC_FQ25GN9BKN > 관리하기 > 청소하기 > 클린봇 기능 설정하기(옵션)
      본 내용은 클린봇 기능이 있는 모델에 적용됩니다. 바람이 들어오는 극세 필터를 클린봇으로 간편하게 청소할  수 있습니다. 1 리모컨의 전원 버튼을 눌러 제품을 켜세요. 2 리모컨의 설정 버튼을 누르세요. 3 리모컨의 L M 버튼을 눌러 클린봇을 선택한 다음  리모컨의 ...

  [2] distance=0.621  AC_FQ25GN9BKN > 고장 신고 전 확인하기 > 클린봇(옵션)
      증상: 클린봇이 작동하지 않아요. 극세 필터가 제대로 장착되어 있나요? • 극세 필터가 제품 뒷면에 제대로 장착되어 있는지 확인하세요. 클린봇의 자동 청소 기능이 해제되어 있나요? • 리모컨의 설정 버튼을 눌러 클린봇을 선택한 다음 기능을 켜세요. 누적 운전 시간이 5...

  [3] distance=0.802  AC_FQ25GN9BKN > 관리하기 > 청소하기 > 먼지통 비움 알림 초기화하기(옵션)
      본 내용은 클린봇 기능이 있는 모델에 적용됩니다. 클린봇이 작동한 횟수를 고려하여 먼지통 비우는 시기를  알려줍니다. 먼지통을 비워야 하는 시기가 되면 실내기  표시부에 안내 문구가 나타납니다. 먼지통을 비우거나 청소한 후 먼지통 비움 알림을  초기화하세요. ~ 제품 ...

Q: 클린봇 기능 어떻게 써요?  [모델: AC_FQ16FV6EDN]


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.12it/s]


  [1] distance=0.985  AC_FQ16FV6EDN > 리모컨으로 사용하기 > 냉방 기본 기능 작동하기 > 공기청정 기능 사용하기
      1 리모컨의 전원 버튼을 눌러 제품을 켜세요. 2 리모컨의 공기청정 버튼을 누르세요. 알아두기 • 공기청정과 냉방, 제습 운전을 동시에 사용할 수  있습니다. - 냉방, 제습 운전 중에 리모컨의 공기청정 버튼을  누르면 실내기 표시부에 공기청정 설정이 나타나고  해당 ...

  [2] distance=1.006  AC_FQ16FV6EDN > 관리하기 > 청소하기 > 열교환기 세척 기능 설정하기
      제품 내부에 있는 열교환기를 얼리고 녹이는 과정을  반복하여 먼지 같은 오염 물질을 제거합니다. 켜짐 예약 또는 꺼짐 예약이 설정되어 있다면 예약 기능을  해제한 다음 열교환기 세척 기능을 사용하세요. ~ 제품  사용설명서의 리모컨으로 사용하기(예약 취소하기)를  보세...

  [3] distance=1.012  AC_FQ16FV6EDN > 관리하기 > 청소하기 > AI 건조 설정하기
      냉방 또는 제습 운전을 하다가 제품의 전원을 끌 경우 AI  건조를 시작합니다. 일정 시간 동안 제품을 송풍 상태로  운전하여 열교환기에 남은 수분을 제거합니다. 1 리모컨의 전원 버튼을 눌러 제품을 켜세요. 2 리모컨의 설정 버튼을 누르세요. 3 리모컨의 L M 버튼...


### 자체 테스트셋으로 정확도 측정

`testset.json` — 등록된 모델 + 구어체 질문 → 정답 청크 63케이스
(howto 18 / care 11 / symptom 17 / error_code 7 / safety 3 / info 3 / 기능 없음 4).
정답은 청크의 `subsection` 라벨 또는 `body` 문구로 지정한다. 케이스를 추가하려면 JSON에 한 줄 넣고 이 셀만 재실행.

기준(3모델 + 에러코드, bge-m3): **top-1 83% / hit@3 92% / MRR 0.86**. 1등이 아닌 10개는 출력 하단에 후보와 함께 나온다.


In [24]:
# ── 자체 테스트셋으로 검색 정확도 측정 ──
# testset.json: "등록된 모델 + 구어체 질문 → 정답 청크" 케이스 모음.
# COMMON_QUESTIONS(팀 공용, 여러 제품군 섞임)와 달리 LG 에어컨 범위 안에서 검색이 정확한지만 본다.
#   - top1  : 1등이 정답인 비율
#   - hit@k : top_k 안에 정답이 있는 비율 (LLM에 정답이 전달되는지)
#   - MRR   : 정답 순위의 역수 평균 (1등=1.0, 2등=0.5, 3등=0.33)
#   - none  : 해당 모델에 없는 기능 질문 → top-1 distance가 NONE_THRESHOLD 이상이면 통과
import json

TESTSET_PATH = Path("testset.json")   # 노트북이 experiments/siyeon/ 안에 있다는 전제
# 정답이 1등일 때 top-1 거리는 0.44~0.94, 해당 기능이 없는 모델에 물었을 때는 0.86~0.98 → 겹치는 구간이 있어서
# 거리만으로 '기능 없음'을 깔끔히 가르진 못한다 (리랭커 확률도 마찬가지). 0.85는 없음 4/4를 잡고 정답 3/49를 오판하는 타협값.
# 최종 판정은 LLM이 근거 문서를 보고 하게 두고, 이 값은 참고 지표로만 본다.
NONE_THRESHOLD = 0.85


def _matches(chunk: dict, expect: dict) -> bool:
    """expect.subsection / expect.body 중 하나라도 맞으면 정답 청크로 본다."""
    if any(s in chunk["subsection"] for s in expect.get("subsection", [])):
        return True
    if any(b in chunk["body"] for b in expect.get("body", [])):
        return True
    return False


def evaluate_testset(path: Path = TESTSET_PATH, top_k: int = 3, none_threshold: float = NONE_THRESHOLD,
                     show_misses: bool = True, search_fn=None, quiet: bool = False) -> list[dict]:
    """search_fn: (query, doc_id=..., top_k=...) -> [(chunk, distance)] 형태면 무엇이든 (기본 search, 고도화 search_v2 등)"""
    search_fn = search_fn or search
    cases = json.loads(path.read_text(encoding="utf-8"))["cases"]
    rows = []
    for case in cases:
        hits = search_fn(case["query"], doc_id=case.get("doc_id"), top_k=top_k)   # doc_id None = 등록 가전 없음
        expect = case["expect"]
        top1_dist = hits[0][1] if hits else float("inf")

        if expect.get("none"):
            rank = None
            ok = top1_dist >= none_threshold
        else:
            rank = next((r for r, (c, _) in enumerate(hits, start=1) if _matches(c, expect)), None)
            ok = rank == 1

        rows.append({
            "id": case["id"], "category": case["category"], "doc_id": case["doc_id"], "query": case["query"],
            "rank": rank, "top1": ok if not expect.get("none") else None, "hit": rank is not None,
            "none_ok": ok if expect.get("none") else None, "top1_distance": round(top1_dist, 3),
            "top1_label": f'{hits[0][0]["section"]} > {hits[0][0]["subsection"]}' if hits else "",
            "top1_body": hits[0][0]["body"].split("\n")[0][:60] if hits else "",
            "candidates": [(f'{c["section"]} > {c["subsection"]}', c["body"].split("\n")[0][:50], round(d, 3)) for c, d in hits],
        })

    normal = [r for r in rows if r["none_ok"] is None]
    nones = [r for r in rows if r["none_ok"] is not None]
    if quiet:
        return rows

    print(f"{'카테고리':12s} {'n':>3s} {'top1':>6s} {'hit@'+str(top_k):>6s} {'MRR':>6s}")
    print("-" * 40)
    for cat in sorted({r["category"] for r in normal}):
        sub = [r for r in normal if r["category"] == cat]
        top1 = sum(r["rank"] == 1 for r in sub) / len(sub)
        hit = sum(r["hit"] for r in sub) / len(sub)
        mrr = sum(1 / r["rank"] for r in sub if r["rank"]) / len(sub)
        print(f"{cat:12s} {len(sub):3d} {top1:6.0%} {hit:6.0%} {mrr:6.2f}")
    print("-" * 40)
    top1 = sum(r["rank"] == 1 for r in normal) / len(normal)
    hit = sum(r["hit"] for r in normal) / len(normal)
    mrr = sum(1 / r["rank"] for r in normal if r["rank"]) / len(normal)
    print(f"{'전체':12s} {len(normal):3d} {top1:6.0%} {hit:6.0%} {mrr:6.2f}")
    if nones:
        passed = sum(r["none_ok"] for r in nones)
        print(f"\n기능 없음 판정(none, threshold={none_threshold}): {passed}/{len(nones)} 통과")
        for r in nones:
            mark = "O" if r["none_ok"] else "X"
            print(f"  [{mark}] {r['query']:40s} top1 dist={r['top1_distance']}  ({r['top1_label']})")

    if show_misses:
        misses = [r for r in normal if r["rank"] != 1]
        if misses:
            print(f"\n=== 1등이 정답이 아닌 케이스 {len(misses)}개 ===")
            for r in misses:
                where = f"{r['rank']}등에 있음" if r["rank"] else f"top{top_k} 안에 없음"
                print(f"\n[{r['id']}] {r['query']}  ({r['doc_id']}) → 정답은 {where}")
                for i, (label, first_line, d) in enumerate(r["candidates"], start=1):
                    flag = "★" if i == r["rank"] else " "
                    print(f"   {flag}{i}. {d:.3f} {label}  |  {first_line}")
    return rows


testset_rows = evaluate_testset(top_k=3)


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.87it/s]

카테고리           n   top1  hit@3    MRR
----------------------------------------
care          11    82%   100%   0.89
error_code     7   100%   100%   1.00
howto         18    83%    94%   0.87
info           3    67%    67%   0.67
safety         3    67%    67%   0.67
symptom       17    82%    88%   0.85
----------------------------------------
전체            59    83%    92%   0.86

기능 없음 판정(none, threshold=0.85): 4/4 통과
  [O] 클린봇 자동 청소 켜는 방법                          top1 dist=0.875  (관리하기 > 청소하기 > AI 건조 설정하기)
  [O] 먼지통은 어떻게 비워요?                            top1 dist=0.975  (관리하기 > 냄새 제거하기 > 냄새에 따른 제거 방법 알아보기)
  [O] 말로 에어컨 조작하려면 어떻게 해요?                     top1 dist=0.856  (리모컨으로 사용하기 > 냉방 기본 기능 작동하기 > 냉방 기능 사용하기)
  [O] 세탁기 탈수가 안 돼요                             top1 dist=0.972  (에러코드 > CH04 / FL (배수 불량, 만수 감지))

=== 1등이 정답이 아닌 케이스 10개 ===

[howto-03] 빨리 시원하게 하는 기능 있나요?  (AC_FQ18GC1EHN) → 정답은 3등에 있음
    1. 0.858 고장 신고 전 확인하기 > 운전  |  증상: 냉방을 켰는데 시원하지 않아요.
    2. 0.895 리모컨으로 사용하기 > 냉방 기본 

## 5-2. 고도화 검색: 질문 확장 + (코드 한정) BM25 + 리랭커

baseline(bge-m3 벡터 검색만)의 1등 실패 10건을 분류해 보니 ① 어휘 간극(`말로 조작`↔`음성인식`) ② 정답이 2~3등 ③ 정확한 단어 매칭 필요(`CH61`, `계속 좋음`) 세 가지였다. 각각에 대응하는 기법을 붙이고 **같은 테스트셋으로 전부 실측**해서 고른 구성:

| 구성 | top-1 | hit@3 | MRR | 초/질문(CPU) |
|---|---|---|---|---|
| baseline (벡터만) | 83% | 92% | 0.86 | 0.05 |
| +리랭커만 | 86~90% | 92% | 0.89 | 3.6 |
| +BM25 항상 (RRF) | **66%** ↓ | 80% | 0.72 | 0.05 |
| +질문확장만 (RRF) | 83% | 95% | 0.88 | 0.10 |
| **질문확장 + 리랭커 (후보 8, 384토큰)** | **88%** | **100%** | **0.94** | 3.6 |

- **리랭커**(bge-reranker-v2-m3): 후보 8개를 크로스인코더가 직접 읽고 재정렬. 가장 큰 효과. 후보 12→8, 길이 512→384가 더 빠르고 더 정확했다.
- **문서 쪽 질문 확장**: 청크마다 gpt-4o-mini로 "물어볼 법한 질문" 4개를 미리 생성해 별도 컬렉션에 임베딩(오프라인 1회, `chunk_questions.json`에 캐시). 어휘 간극 해소. 매 질문마다 LLM을 부르는 쿼리 확장과 달리 응답 시간에 영향 없음.
- **BM25**(kiwi 형태소 분석): 항상 켜면 오히려 -17pt. 질문에 `CH05`처럼 코드가 있을 때만 켠다(`use_bm25="auto"`).

필요 패키지: `kiwipiepy`, `rank_bm25` (requirements.txt에 추가됨). 리랭커는 CPU에서 질문당 ~3.6초라 **서비스에선 GPU 또는 더 작은 리랭커 검토 필요**.


In [25]:
# ── 고도화 검색 1/3: 한국어 BM25 (키워드 검색) ──
# 임베딩은 뜻이 비슷하면 잡아주지만 'CH61', 'NO', 'CL' 같은 코드나 '계속 좋음' 같은
# 정확한 단어 매칭에는 약하다. BM25는 그 반대라서 둘을 섞으면 서로 약점을 메운다.
# 한국어는 띄어쓰기 단위로 자르면 '시원하게/시원해요'가 다른 단어가 되므로 형태소 분석기(kiwi)로 어간을 뽑는다.
import re

import numpy as np
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

_kiwi = Kiwi()
# 내용어만 남긴다: 명사(NN*), 동사(VV), 형용사(VA), 부사(MAG), 영문(SL), 숫자(SN), 어근(XR)
_KEEP_PREFIX = ("NN", "VV", "VA", "MAG", "SL", "SN", "XR")


def tokenize_ko(text: str) -> list[str]:
    toks = []
    for t in _kiwi.tokenize(text):
        if not t.tag.startswith(_KEEP_PREFIX):
            continue
        form = t.form.lower()
        # 한 글자 조사/어미 찌꺼기는 버리되, 영문·숫자(코드)는 한 글자여도 남긴다
        if len(form) == 1 and not t.tag.startswith(("SL", "SN")):
            continue
        toks.append(form)
    return toks


class BM25Index:
    """청크 전체에 대한 BM25. 검색 시 doc_id로 걸러서 top-n을 돌려준다."""

    def __init__(self, chunks: list[dict]):
        self.chunks = chunks
        self.bm25 = BM25Okapi([tokenize_ko(c["text"]) for c in chunks])

    def search(self, query: str, allowed_doc_ids: set[str] | None, n: int) -> list[tuple[dict, float]]:
        scores = self.bm25.get_scores(tokenize_ko(query))
        if allowed_doc_ids is not None:
            mask = np.array([c["doc_id"] in allowed_doc_ids for c in self.chunks])
            scores = np.where(mask, scores, -1.0)
        order = np.argsort(-scores)[:n]
        return [(self.chunks[i], float(scores[i])) for i in order if scores[i] > 0]


bm25_index = BM25Index(chunks)
print(f"BM25 색인 완료: {len(chunks)}개 청크")


BM25 색인 완료: 988개 청크


In [26]:
# ── 고도화 검색 2/3: 문서 쪽 질문 확장 (오프라인 1회) ──
# 실패 케이스의 절반이 '말로 조작'↔'음성인식', '이사'↔'이전 설치' 같은 어휘 간극이었다.
# 질문마다 LLM으로 바꿔 쓰는 대신, 청크마다 "이 내용을 물어볼 법한 질문" 4개를 미리 만들어 두고
# 그 질문들을 임베딩한다. 사용자 질문은 '청크 본문'이 아니라 '비슷한 질문'과 매칭되므로 표현 차이에 강해진다.
# 비용은 청크 수만큼 한 번(1,000개 ≈ 수십 원)이고 서비스 응답 시간엔 영향이 없다.
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

QUESTIONS_CACHE = Path("chunk_questions.json")   # {chunk_id: [질문, ...]} - 생성 결과 캐시 (커밋 대상)
QUESTIONS_PER_CHUNK = 4

_QGEN_PROMPT = """아래는 LG 에어컨 사용설명서의 한 부분이다. 이 내용으로 답할 수 있는, 실제 사용자가 챗봇에 칠 법한 질문을 {n}개 만들어라.

규칙:
- 설명서 용어를 그대로 쓰지 말고 일반 사용자의 구어체로 (예: "음성인식 기능" → "말로 켤 수 있어요?", "이전 설치" → "이사할 때")
- 증상/문제 상황이면 사용자가 겪는 현상 중심으로 (예: "혼자 켜졌어요", "물이 뚝뚝 떨어져요")
- 반드시 아래 [내용]에 실제로 있는 것만 물어라. 내용에 없는 에러코드·기능·상황을 지어내지 마라
{code_rule}- 짧게, 한 문장씩. JSON 배열만 출력: ["질문1", "질문2", ...]

[섹션] {label}
[내용]
{body}"""


def generate_questions_for_chunk(client: OpenAI, c: dict) -> list[str]:
    label = f'{c["section"]} > {c["subsection"]}' if c["subsection"] else c["section"]
    # 에러코드 청크에만 "코드 포함 질문" 규칙을 넣는다 (다른 청크에 넣으면 없는 에러코드 질문을 지어냄)
    code_rule = "- 에러코드(CH04 등)를 그대로 포함한 질문 1개 이상\n" if c["tag"] == "error_code" else ""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.7,
        messages=[{"role": "user", "content": _QGEN_PROMPT.format(
            n=QUESTIONS_PER_CHUNK, label=label, body=c["body"][:1500], code_rule=code_rule)}],
    )
    text = resp.choices[0].message.content.strip()
    text = text[text.find("["): text.rfind("]") + 1]   # 코드펜스 등 잡음 제거
    qs = json.loads(text)
    return [q.strip() for q in qs if isinstance(q, str) and q.strip()][:QUESTIONS_PER_CHUNK]


def build_question_cache(chunks: list[dict], cache_path: Path = QUESTIONS_CACHE, max_workers: int = 8) -> dict[str, list[str]]:
    """캐시에 없는 청크만 생성해서 캐시 파일을 갱신하고 전체 dict를 돌려준다."""
    cache = json.loads(cache_path.read_text(encoding="utf-8")) if cache_path.exists() else {}
    todo = [c for c in chunks if c["id"] not in cache]
    if not todo:
        print(f"질문 캐시 재사용: {len(cache)}개 청크")
        return cache
    api_key = os.getenv("OPENAI_API_KEY", "")
    if not api_key:
        print(f"[경고] OPENAI_API_KEY 없음 - 질문 생성 건너뜀 (캐시 {len(cache)}개만 사용)")
        return cache
    client = OpenAI(api_key=api_key)
    print(f"질문 생성: {len(todo)}개 청크 (캐시 {len(cache)}개 있음) ...")
    failed = 0
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(generate_questions_for_chunk, client, c): c["id"] for c in todo}
        for i, fut in enumerate(as_completed(futures), start=1):
            try:
                cache[futures[fut]] = fut.result()
            except Exception as e:  # JSON 파싱 실패/일시적 API 오류 - 다음 실행에서 재시도됨
                failed += 1
            if i % 100 == 0 or i == len(todo):
                cache_path.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding="utf-8")
                print(f"  {i}/{len(todo)} (실패 {failed})")
    return cache


def build_question_store(vectorstore, chunks: list[dict], cache: dict[str, list[str]]):
    """질문 하나 = 문서 하나로 별도 컬렉션에 임베딩. metadata.parent_id로 원래 청크를 찾아간다.
    본문 컬렉션과 분리해 두면 기존 search()에 질문 문서가 섞여 들어가지 않는다."""
    from langchain_chroma import Chroma
    qstore = Chroma(collection_name="chunk_questions", embedding_function=vectorstore.embeddings, client=vectorstore._client)
    existing = qstore.get(include=[])["ids"]
    if existing:
        qstore.delete(ids=existing)
    docs, ids = [], []
    for c in chunks:
        for i, q in enumerate(cache.get(c["id"], [])):
            docs.append(Document(page_content=q, metadata={"parent_id": c["id"], "doc_id": c["doc_id"]}))
            ids.append(f'{c["id"]}__q{i}')
    if docs:
        qstore.add_documents(docs, ids=ids)
    print(f"질문 문서 {len(docs)}개 임베딩 완료 (청크 {sum(1 for c in chunks if cache.get(c['id']))}개)")
    return qstore


question_cache = build_question_cache(chunks)
question_store = build_question_store(vectorstore, chunks, question_cache)


질문 생성: 691개 청크 (캐시 297개 있음) ...
  100/691 (실패 0)
  200/691 (실패 0)
  300/691 (실패 0)
  400/691 (실패 0)
  500/691 (실패 0)
  600/691 (실패 0)
  691/691 (실패 0)


Batches: 100%|██████████| 124/124 [01:30<00:00,  1.37it/s]


질문 문서 3952개 임베딩 완료 (청크 988개)


In [27]:
# ── 고도화 검색 3/3: RRF 융합 + 리랭커 ──
# 세 검색기(벡터-본문, 벡터-질문, BM25)의 순위를 RRF로 합쳐 후보 n개를 만들고,
# 크로스인코더 리랭커가 (질문, 청크) 쌍을 직접 읽고 다시 정렬한다.
# 리랭커는 임베딩보다 훨씬 정밀하지만 느려서(쌍마다 모델 통과) 후보 10~15개에만 쓴다.
from sentence_transformers import CrossEncoder

# 다국어 리랭커(로컬). max_length 384: 512보다 빠르고 테스트셋에서 정확도도 더 좋았다 (후보 잡음이 줄어서).
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=384)
# 질문에 에러/표시 코드가 들어 있는지 (CH05, ch 61, E6, F4 …). 이때만 BM25를 켠다.
CODE_RE = re.compile(r"\b[A-Za-z]{1,2}\s?-?\d{1,3}\b")
CHUNK_BY_ID = {c["id"]: c for c in chunks}
RRF_K = 60


def _allowed(doc_id) -> set[str] | None:
    if doc_id is None:
        return None
    ids = [doc_id] if isinstance(doc_id, str) else list(doc_id)
    _as_filter(doc_id)   # 모델명 검증(없는 모델이면 ValueError)
    return set(ids) | {COMMON_DOC_ID}


def _rrf(ranked_lists: list[list[str]]) -> list[str]:
    """여러 순위 목록을 Reciprocal Rank Fusion으로 합친다. 여러 검색기에서 상위에 나온 청크가 위로 온다."""
    score: dict[str, float] = {}
    for ranked in ranked_lists:
        for rank, cid in enumerate(ranked, start=1):
            score[cid] = score.get(cid, 0.0) + 1.0 / (RRF_K + rank)
    return sorted(score, key=score.get, reverse=True)


def search_v2(query: str, doc_id=None, top_k: int = 3, n_candidates: int = 8,
              use_bm25: bool | str = "auto", use_questions: bool = True, use_rerank: bool = True) -> list[tuple[dict, float]]:
    """search()와 같은 (chunk_dict, distance) 형식. distance는 낮을수록 좋음 (리랭커 사용 시 1 - 관련확률).

    테스트셋 63케이스 기준 (baseline 벡터만: top1 83% / hit@3 92%):
      - 리랭커만                      top1 86~90%
      - 질문확장 + 리랭커 (n=8)        top1 88% / hit@3 100%   ← 기본값
      - BM25 항상 켜면                top1 66%로 오히려 악화 → use_bm25="auto": 질문에 코드가 있을 때만
      - n_candidates 12→8, 길이 512→384가 더 빠르고 더 정확했다
    주의: 리랭커 확률도 임베딩 거리처럼 정답/기능없음이 겹쳐서 "해당 기능 없음" 판정에는 못 쓴다.
    """
    if use_bm25 == "auto":
        use_bm25 = bool(CODE_RE.search(query))
    allowed = _allowed(doc_id)
    filt = _as_filter(doc_id)
    ranked_lists: list[list[str]] = []

    # 1) 벡터 - 본문
    hits = vectorstore.similarity_search_with_score(query, k=n_candidates, filter=filt)
    ranked_lists.append([d.metadata["id"] for d, _ in hits])
    vec_dist = {d.metadata["id"]: float(s) for d, s in hits}

    # 2) 벡터 - 미리 생성한 질문 → 부모 청크 (같은 부모가 여러 번 나오면 첫 등장만)
    if use_questions and question_store is not None:
        qhits = question_store.similarity_search_with_score(query, k=n_candidates * 2, filter=filt)
        seen, parents = set(), []
        for d, s in qhits:
            pid = d.metadata["parent_id"]
            if pid not in seen and pid in CHUNK_BY_ID:
                seen.add(pid); parents.append(pid)
                vec_dist.setdefault(pid, float(s))
        ranked_lists.append(parents[:n_candidates])

    # 3) BM25
    if use_bm25:
        ranked_lists.append([c["id"] for c, _ in bm25_index.search(query, allowed, n_candidates)])

    fused = [cid for cid in _rrf(ranked_lists) if cid in CHUNK_BY_ID][:n_candidates]

    if not use_rerank:
        # 리랭커 없이 RRF 순위만 쓸 때: distance는 벡터 거리(없으면 RRF 순서 기준 자리값)
        return [(CHUNK_BY_ID[cid], vec_dist.get(cid, 1.0 + i * 0.01)) for i, cid in enumerate(fused[:top_k])]

    # 4) 리랭커: (질문, 청크 text) 쌍을 직접 읽고 점수화
    pairs = [(query, CHUNK_BY_ID[cid]["text"]) for cid in fused]
    probs = np.asarray(reranker.predict(pairs))              # 0~1 관련 확률 (sigmoid 적용됨). 클수록 관련
    order = np.argsort(-probs)[:top_k]
    return [({**CHUNK_BY_ID[fused[i]], "rerank_score": float(probs[i])}, float(1.0 - probs[i])) for i in order]


Loading weights: 100%|██████████| 393/393 [00:01<00:00, 301.71it/s]


### baseline vs 고도화 비교

같은 테스트셋을 `search`(baseline)와 `search_v2`로 각각 돌려 비교한다. 리랭커 때문에 63케이스 × 3.6초 ≈ 4분.


In [28]:
import time

def summarize(rows):
    normal = [r for r in rows if r["none_ok"] is None]
    return (sum(r["rank"] == 1 for r in normal) / len(normal),
            sum(r["hit"] for r in normal) / len(normal),
            sum(1 / r["rank"] for r in normal if r["rank"]) / len(normal))

configs = {
    "baseline (벡터만)":                 search,
    "질문확장 + 코드한정 BM25 + 리랭커":  search_v2,
}
compare_rows = {}
print(f"{'구성':34s} {'top1':>6s} {'hit@3':>6s} {'MRR':>6s} {'초/질문':>7s}")
for name, fn in configs.items():
    t = time.time()
    rows = evaluate_testset(top_k=3, search_fn=fn, quiet=True)
    compare_rows[name] = rows
    top1, hit, mrr = summarize(rows)
    print(f"{name:34s} {top1:6.0%} {hit:6.0%} {mrr:6.2f} {(time.time() - t) / len(rows):7.2f}")

# 고도화 구성에서 1등이 아닌 케이스 상세
print("\n=== search_v2 1등 실패 ===")
for r in compare_rows["질문확장 + 코드한정 BM25 + 리랭커"]:
    if r["none_ok"] is None and r["rank"] != 1:
        print(f"[{r['id']}] {r['query']} → 정답 {r['rank'] or 'top3 밖'}등")
        for i, (label, first_line, d) in enumerate(r["candidates"], start=1):
            print(f"   {'★' if i == r['rank'] else ' '}{i}. {d:.3f} {label} | {first_line}")


구성                                   top1  hit@3    MRR    초/질문


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.01it/s]


baseline (벡터만)                        83%    92%   0.86    0.06


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.41it/s]


질문확장 + 코드한정 BM25 + 리랭커                88%   100%   0.94    3.71

=== search_v2 1등 실패 ===
[howto-13] 스마트폰 앱에 에어컨 등록하는 방법 → 정답 3등
    1. 0.886 리모컨으로 사용하기 > 리모컨 살펴보기 > 리모컨 처음 사용하기 | 리모컨 버튼 살펴보기
    2. 0.932 조작부로 사용하기 > 조작부 살펴보기 > 실내기 버튼부 | 리모컨 없이도 실내기 버튼부에서 전원을 켜고 끄거나 희망하는 온도와 바람 세기를 설정할 수
   ★3. 0.955 LG ThinQ 사용하기 > LG ThinQ와 LG 가전 연결하기 > 앱 설치 및 제품 등록하기 | LG ThinQ 앱을 설치하면 언제 어디서나 편리하게 우리집 
[care-05] 에어컨 끄면 한동안 바람이 계속 나와요. 이거 뭐예요? → 정답 2등
    1. 0.617 고장 신고 전 확인하기 > 운전 | 증상: 차가운 바람이 나오다가 멈춰요.
   ★2. 0.887 고장 신고 전 확인하기 > 운전 | 증상: 전원을 껐는데도 계속 작동해요.
    3. 0.976 리모컨으로 사용하기 > 리모컨 살펴보기 > 리모컨 처음 사용하기 | 바람세기 + -
[care-09] 먼지통은 어떻게 비워요? → 정답 2등
    1. 0.014 관리하기 > 청소하기 > 먼지통 비움 알림 초기화하기(옵션) | 본 내용은 클린봇 기능이 있는 모델에 적용됩니다.
   ★2. 0.625 관리하기 > 청소하기 > 먼지통 청소하기(옵션) | 본 내용은 클린봇 기능이 있는 모델에 적용됩니다.
    3. 0.999 관리하기 > 냄새 제거하기 > 냄새에 따른 제거 방법 알아보기 | 제품에서 이상한 냄새가 날 때는 아래 방법을 참고하여 
[symptom-01] 전원 버튼 눌러도 안 켜져요 → 정답 2등
    1. 0.701 고장 신고 전 확인하기 > 운전 | 증상: 실내기의 좌우 토출구가 닫히지 않아요. (모델에 따라 좌우 토출구 덮개가 없을 수 
   ★2. 0

### 5-3. RDB 라우터 (search_v3) — 에러코드·기능 유무는 검색이 아니라 조회로

먼저 한 번 실행: `python experiments/siyeon/rdb/build_db.py --check` → `data/appliance.sqlite` 생성
(스키마·설계 이유는 `experiments/siyeon/rdb/schema.sql`).

```
등록 가전 (brand, category, model, manual_id)
 ① 질문에 코드(CH05, ch 61, 0E, FF …) → error_codes 조회 → 히트면 1등 고정 (0ms, 리랭커 불필요). F4처럼 두 항목이면 둘 다
 ② 질문에 기능명(클린봇, 말로 …)      → model_features 표 → 없으면 검색 생략하고 "이 모델엔 없는 기능" 응답
                                        표에 모델이 없으면(SQ 벽걸이, GC1) 매뉴얼 본문에 그 단어가 0회인지로 2차 판정
 ③ 나머지 → search_v2. 에러 얘기가 아닌 질문엔 에러코드 청크 제외 ('혼자 켜졌어요'를 CH237이 가로채던 문제)
```

이걸로 "기능 없음" 판정이 점수 임계값(불안정)에서 **룰**로 바뀐다.


In [33]:
# ── 고도화 검색 4/4: RDB 라우터 (search_v3) ──
# 벡터 검색 앞에 두 개의 결정적(deterministic) 단계를 둔다. 둘 다 SQLite 조회라 ms 단위.
#   ① 질문에 에러코드가 있으면  → error_codes 테이블 조회 → 히트면 그 항목을 1등으로 고정 (정확도 100%, 리랭커 불필요)
#   ② 질문에 기능명이 있으면    → model_features 표 → 이 모델에 없는 기능이면 검색을 생략하고 "없음"을 답으로
#      표에 모델이 없으면(SQ 벽걸이형, GC1 등) → 그 매뉴얼 청크에 기능명이 한 번도 안 나오는지로 2차 판정
#   ③ 나머지는 search_v2. 단, 에러 얘기가 아닌 질문(코드도 없고 '에러/오류/코드/점검/표시' 같은 단어도 없음)에는
#      에러코드 청크를 후보에서 뺀다 ('혼자 켜졌어요' 같은 증상 질문을 CH237 에러코드 청크가 가로채는 걸 실측했음).
#      '통신 에러래요'처럼 코드 없이 말로 설명한 에러 질문은 단어로 감지해서 에러코드 청크를 남긴다.
# DB는 experiments/siyeon/rdb/build_db.py 로 생성 (data/appliance.sqlite). 코드 정규화 함수도 거기서 가져와 같은 규칙을 쓴다.
import sqlite3
import sys

sys.path.insert(0, str(Path.cwd().resolve() / "rdb"))
from build_db import code_variants, extract_query_codes, FEATURE_IMPLIED_BY  # noqa: E402

DB_PATH = Path("../../data/appliance.sqlite")
db = sqlite3.connect(DB_PATH, check_same_thread=False)
db.row_factory = sqlite3.Row

_FEATURE_ALIASES = [(r["alias"], r["feature"]) for r in db.execute("SELECT alias, feature FROM feature_aliases ORDER BY length(alias) DESC")]
_FEATURE_KEYWORDS = {r["feature"]: [r["alias"] for r in db.execute("SELECT alias FROM feature_aliases WHERE feature = ?", (r["feature"],))]
                     for r in db.execute("SELECT DISTINCT feature FROM feature_aliases")}


def appliance_for(doc_id: str) -> dict:
    """테스트용: 청크 doc_id → 등록 가전 dict. 웹에서는 user_appliances 행이 이 역할."""
    row = db.execute("SELECT brand, category FROM manuals WHERE manual_id = ?", (doc_id,)).fetchone()
    if row is None:
        raise ValueError(f"manuals 테이블에 없는 매뉴얼: {doc_id}")
    return {"brand": row["brand"], "category": row["category"], "model": doc_id.split("_", 1)[1].upper(), "manual_id": doc_id}


def lookup_error_codes(brand: str, category: str, query: str) -> list[dict]:
    """질문 속 코드 → RDB 조회. 등록 가전의 (brand, category)로 범위를 좁힌다 (FF가 세탁기/냉장고에 다 있음)."""
    variants = sorted({v for tok in extract_query_codes(query) for v in code_variants(tok)})
    if not variants:
        return []
    rows = db.execute(
        f"SELECT e.entry_id, e.title, e.summary, e.content, e.chunk_id, c.code_display, c.kind "
        f"FROM error_codes c JOIN error_entries e USING (entry_id) "
        f"WHERE c.brand = ? AND c.category = ? AND c.code_norm IN ({','.join('?' * len(variants))}) "
        f"ORDER BY c.kind = 'primary' DESC, e.entry_id",
        (brand, category, *variants),
    ).fetchall()
    seen, out = set(), []
    for r in rows:
        if r["entry_id"] not in seen:
            seen.add(r["entry_id"]); out.append(dict(r))
    return out


def detect_feature(query: str) -> str | None:
    q = re.sub(r"\s+", "", query).lower()
    for alias, feature in _FEATURE_ALIASES:        # 긴 별칭부터 (부분 문자열 오탐 방지)
        if alias in q:
            return feature
    return None


def feature_available(appliance: dict, feature: str) -> bool | None:
    """True 있음 / False 없음 / None 모름."""
    target = FEATURE_IMPLIED_BY.get(feature, feature)   # 먼지통 → 클린봇 유무로 판정
    row = db.execute(
        "SELECT has FROM model_features WHERE brand = ? AND category = ? AND feature = ? "
        "AND ? GLOB replace(model_pattern, '*', '?')",
        (appliance["brand"], appliance["category"], target, appliance["model"]),
    ).fetchone()
    if row is not None:
        return bool(row["has"])
    # 2차: 표에 모델이 없으면 매뉴얼 본문에 기능명이 한 번이라도 나오는지 (0회면 없는 기능으로 본다)
    if appliance.get("manual_id"):
        kws = [feature.lower()] + _FEATURE_KEYWORDS.get(feature, [])
        for c in chunks:
            if c["doc_id"] == appliance["manual_id"]:
                hay = re.sub(r"\s+", "", (c["subsection"] + " " + c["body"])).lower()
                if any(k in hay for k in kws):
                    return None                      # 언급은 있음 → 모름, 검색으로
        return False
    return None


# 코드는 없지만 에러/점검 관련 질문임을 알리는 단어 → 에러코드 청크를 후보에 남김
ERROR_WORDS_RE = re.compile(r"에러|오류|코드|점검|표시|떴|떠요|뜨는|뜬다|경고|고장")


def _error_entry_as_chunk(e: dict) -> dict:
    """RDB 항목을 후보(chunk dict) 형식으로. 벡터 인덱스에 같은 청크가 있으면 그 id를 그대로 쓴다."""
    c = CHUNK_BY_ID.get(e["chunk_id"])
    if c:
        return {**c, "source": "rdb"}
    body = e["content"]
    return {"id": e["chunk_id"] or f"rdb_error_{e['entry_id']}", "doc_id": "RDB", "section": "에러코드",
            "subsection": e["title"], "body": body, "tag": "error_code", "text": f"에러코드 > {e['title']}\n{body}", "source": "rdb"}


def search_v3(query: str, doc_id=None, top_k: int = 3, appliance: dict | None = None) -> list[tuple[dict, float]]:
    """라우터. 평가 호환을 위해 doc_id(=manual_id)로도 부를 수 있다. 반환 형식은 search()와 동일."""
    if appliance is None:
        if doc_id is None:
            return search_v2(query, None, top_k)         # 가전 미지정 → 전체 벡터 검색
        appliance = appliance_for(doc_id if isinstance(doc_id, str) else doc_id[0])
    manual_id = appliance.get("manual_id")

    # ① 에러코드
    entries = lookup_error_codes(appliance["brand"], appliance["category"], query)
    if entries:
        hits = [(_error_entry_as_chunk(e), 0.0) for e in entries[:top_k]]
        if len(hits) < top_k and manual_id:             # 남는 자리는 매뉴얼의 관련 증상으로 보강 (에러코드 청크 제외)
            extra = [(c, d) for c, d in search_v2(query, manual_id, top_k + 3, use_bm25=False)
                     if c["tag"] != "error_code" and c["id"] not in {h[0]["id"] for h in hits}]
            hits += extra[: top_k - len(hits)]
        return hits

    # ② 기능 유무
    feature = detect_feature(query)
    if feature and manual_id:
        avail = feature_available(appliance, feature)
        if avail is False:
            body = f"이 모델({appliance['model']})에는 '{feature}' 기능이 없습니다. (모델별 기능 표/매뉴얼 기준)"
            return [({"id": f"feature_absent_{feature}", "doc_id": manual_id, "section": "기능 없음", "subsection": feature,
                      "body": body, "tag": "feature_absent", "text": body, "source": "rdb"}, 1.0)]

    # ③ 일반 검색 — 에러 얘기가 아닌 질문엔 에러코드 청크 제외 (에러 단어가 있으면 유지)
    hits = search_v2(query, manual_id, top_k + 3)
    if not ERROR_WORDS_RE.search(query):
        filtered = [(c, d) for c, d in hits if c["tag"] != "error_code"]
        if filtered:
            hits = filtered
    return hits[:top_k]


In [34]:
# 라우터 동작 확인 — 세 경로(①코드 ②기능없음 ③일반)가 각각 어디서 답을 가져오는지. source 열: rdb / vec
import time

for q, d in [
    ("ch 61 에러", "AC_SQ09GK1WEN"),                          # ① 코드 → RDB, 표기 흔들려도(소문자·공백) 잡힘
    ("F4 에러 떴어요", "AC_FQ18GC1EHN"),                      # ① 한 코드가 두 항목(CH38, CH90/91) → 둘 다
    ("클린봇 자동 청소 켜는 방법", "AC_FQ18GC1EHN"),            # ② 이 모델엔 없음 → 검색 생략
    ("클린봇 자동 청소 켜는 방법", "AC_FQ25GN9BKN"),            # ② 있음 → ③ 일반 검색으로
    ("아무것도 안 했는데 에어컨이 혼자 켜졌어요", "AC_FQ18GC1EHN"),  # ③ 에러코드 청크 제외 → 증상 행이 1등
    ("실내기랑 실외기 통신 에러래요", "AC_FQ18GC1EHN"),          # ③ 코드는 없지만 '에러' 단어 → 에러코드 청크 유지
]:
    t = time.time()
    hits = search_v3(q, doc_id=d)
    print(f"\nQ: {q}  [{d}]  ({time.time() - t:.2f}s)")
    for c, dist in hits:
        print(f"   {dist:.3f} {c.get('source', 'vec'):3s} {c['section']} > {c['subsection'][:50]}")


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.13it/s]



Q: ch 61 에러  [AC_SQ09GK1WEN]  (3.23s)
   0.000 rdb 에러코드 > CH61 (실외기/실내기 온도 이상, 제품에 따라 P4/P6/P7/P8/CH34)


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.90it/s]



Q: F4 에러 떴어요  [AC_FQ18GC1EHN]  (3.84s)
   0.000 rdb 에러코드 > CH38 / F4 (냉매 가스 부족)
   0.000 rdb 에러코드 > CH90 / CH91 / F4 (시운전 보호 코드)
   0.999 vec 안전을 위해 주의하기 > 경고 > 제품 이상 및 고장이 발생했을 때

Q: 클린봇 자동 청소 켜는 방법  [AC_FQ18GC1EHN]  (0.00s)
   1.000 rdb 기능 없음 > 클린봇


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.79it/s]



Q: 클린봇 자동 청소 켜는 방법  [AC_FQ25GN9BKN]  (3.96s)
   0.001 vec 관리하기 > 청소하기 > 클린봇 기능 설정하기(옵션)
   0.004 vec 고장 신고 전 확인하기 > 클린봇(옵션)
   0.565 vec 관리하기 > 청소하기 > 먼지통 비움 알림 초기화하기(옵션)


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.57it/s]



Q: 아무것도 안 했는데 에어컨이 혼자 켜졌어요  [AC_FQ18GC1EHN]  (3.76s)
   0.978 vec 고장 신고 전 확인하기 > 운전
   0.981 vec 고장 신고 전 확인하기 > 운전
   0.990 vec 고장 신고 전 확인하기 > 운전


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.71it/s]



Q: 실내기랑 실외기 통신 에러래요  [AC_FQ18GC1EHN]  (3.25s)
   0.006 vec 에러코드 > CH93 (실내기·실외기 통신 이상)
   0.006 vec 에러코드 > CH05 / E0 / CH53 (통신 에러)
   0.014 vec 에러코드 > CH66 (통신선 결선/배관 연결 이상)


In [35]:
# 세 단계 최종 비교: baseline → search_v2 → search_v3.  리랭커 때문에 63케이스 × 2구성 ≈ 8분
configs = {
    "baseline (벡터만)":               search,
    "v2: 질문확장+코드BM25+리랭커":      search_v2,
    "v3: + RDB 라우터":                search_v3,
}
final_rows = {}
print(f"{'구성':30s} {'top1':>6s} {'hit@3':>6s} {'MRR':>6s} {'none':>6s} {'초/질문':>7s}")
for name, fn in configs.items():
    t = time.time()
    rows = evaluate_testset(top_k=3, search_fn=fn, quiet=True)
    final_rows[name] = rows
    normal = [r for r in rows if r["none_ok"] is None]
    nones = [r for r in rows if r["none_ok"] is not None]
    top1 = sum(r["rank"] == 1 for r in normal) / len(normal)
    hit = sum(r["hit"] for r in normal) / len(normal)
    mrr = sum(1 / r["rank"] for r in normal if r["rank"]) / len(normal)
    print(f"{name:30s} {top1:6.0%} {hit:6.0%} {mrr:6.2f} {sum(r['none_ok'] for r in nones):>3d}/{len(nones):<2d} {(time.time() - t) / len(rows):7.2f}")

print("\n=== v3 1등 실패 ===")
for r in final_rows["v3: + RDB 라우터"]:
    if r["none_ok"] is None and r["rank"] != 1:
        print(f"[{r['id']}] {r['query']} → 정답 {r['rank'] or 'top3 밖'}등 | top1: {r['top1_label']}")


구성                               top1  hit@3    MRR   none    초/질문


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.00it/s]


baseline (벡터만)                    83%    92%   0.86   4/4     0.06


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.38it/s]


v2: 질문확장+코드BM25+리랭커               88%   100%   0.94   4/4     3.84


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.77it/s]


v3: + RDB 라우터                     90%   100%   0.95   4/4     3.61

=== v3 1등 실패 ===
[howto-13] 스마트폰 앱에 에어컨 등록하는 방법 → 정답 3등 | top1: 리모컨으로 사용하기 > 리모컨 살펴보기 > 리모컨 처음 사용하기
[care-05] 에어컨 끄면 한동안 바람이 계속 나와요. 이거 뭐예요? → 정답 2등 | top1: 고장 신고 전 확인하기 > 운전
[care-09] 먼지통은 어떻게 비워요? → 정답 2등 | top1: 관리하기 > 청소하기 > 먼지통 비움 알림 초기화하기(옵션)
[symptom-01] 전원 버튼 눌러도 안 켜져요 → 정답 2등 | top1: 고장 신고 전 확인하기 > 운전
[safety-01] 에어컨에서 타는 냄새가 나는데 계속 써도 돼요? → 정답 2등 | top1: 관리하기 > 냄새 제거하기 > 냄새에 따른 제거 방법 알아보기
[safety-02] 집에 가스 냄새 나는데 에어컨 켜도 돼요? → 정답 2등 | top1: 알아보기 > 에어컨의 모습과 기능 살펴보기 > 사용하기 전 알아두기


## 6. 팀 평가 질문(MODEL_QUESTIONS)으로 검색 품질 확인

팀 공용 채점 모듈 `pipeline/evaluate.py`는 2026-09에 `COMMON_QUESTIONS`(22개) → **`MODEL_QUESTIONS`(60모델 × 3라운드 × 5문항 = 900개)** 로
바뀌었다. 질문에 모델명이 들어가므로(`FQ18GC1EHN 에어컨 필터 청소 방법 알려줘`) 등록 가전 없이도 `search_v3`가 모델명을 뽑아 필터를 건다.

여기서는 LLM 없이 **검색 단계만** 채점한다(candidates에 키워드가 있는지). 이 노트북 인덱스는 LG 에어컨만이라
LG 에어컨 150문항만 돌린다. 전 제품군 정식 채점은 `python experiments/siyeon/exp/exp03_lg.py` (900문항, ~80분).


In [ ]:
import sys
from pathlib import Path

# 노트북이 experiments/siyeon/ 안에 있다고 가정. 프로젝트 루트를 path에 추가.
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))

from pipeline.evaluate import MODEL_QUESTIONS, SYSTEM_PROMPT   # (question, keyword, model, description, round)

# 이 노트북 인덱스에 들어 있는 모델(LG 에어컨)의 질문만
_indexed_models = {d.split("_", 1)[1] for d in known_doc_ids}
EVAL_QUESTIONS = [q for q in MODEL_QUESTIONS if q[2] in _indexed_models]
print(f"MODEL_QUESTIONS {len(MODEL_QUESTIONS)}개 중 이 인덱스 범위 {len(EVAL_QUESTIONS)}개")


In [ ]:
def check_retrieval_only(search_fn=None, top_k: int = 3, questions=None) -> list[dict]:
    """EVAL_QUESTIONS에 대해 검색만 돌려서 정답 키워드가 candidates 안에 있는지 확인. LLM 답변은 생성 안 함.
    search_fn 기본은 search_v3 (질문 속 모델명으로 가전을 해석해 필터). 리랭커 때문에 150문항 ≈ 9분."""
    search_fn = search_fn or search_v3
    rows = []
    for question, keyword, model, description, round_name in (questions or EVAL_QUESTIONS):
        hits = search_fn(question, top_k=top_k)
        candidate_text = " ".join(chunk["text"] for chunk, _ in hits)
        keyword_hit = keyword in candidate_text
        rows.append({
            "question": question, "keyword": keyword, "model": model, "round": round_name, "description": description,
            "keyword_hit": keyword_hit,
            "top1_section": f"{hits[0][0]['doc_id']} > {hits[0][0]['section']} > {hits[0][0]['subsection']}" if hits else "",
            "top1_distance": round(hits[0][1], 3) if hits else None,
        })
        print(f"[{'O' if keyword_hit else 'X'}] {description:32s} | {question}")
    return rows


retrieval_check = check_retrieval_only(search_fn=search_v3, top_k=3)
hit = sum(r["keyword_hit"] for r in retrieval_check)
print(f"\n검색 단계 키워드 히트율: {hit / len(retrieval_check):.0%}  ({hit}/{len(retrieval_check)})")
for rn in sorted({r["round"] for r in retrieval_check}):
    sub = [r for r in retrieval_check if r["round"] == rn]
    print(f"  {rn}: {sum(r['keyword_hit'] for r in sub) / len(sub):.0%}")


In [ ]:
# 히트 실패한 질문만 따로 모아서 원인 살펴보기 (키워드 채점의 한계도 같이 보인다: '무풍'은 삼성 용어라 LG 문서엔 없음)
misses = [r for r in retrieval_check if not r["keyword_hit"]]
from collections import Counter
print("실패 키워드 분포:", Counter(r["keyword"] for r in misses))
for r in misses:
    print(f"- [{r['description']}] {r['question']}  (키워드: '{r['keyword']}')")
    print(f"    top1 -> {r['top1_section']} (distance={r['top1_distance']})")


In [32]:
from langchain_chroma import Chroma
import chromadb
client = chromadb.PersistentClient(path="../../chroma_lg_aircon")
for name, src in [("langchain", vectorstore), ("chunk_questions", question_store)]:
    data = src.get(include=["embeddings", "documents", "metadatas"])
    col = client.get_or_create_collection(name)
    col.add(ids=data["ids"], embeddings=data["embeddings"], documents=data["documents"], metadatas=data["metadatas"])
print("저장 완료:", [c.name for c in client.list_collections()])
# 다음 세션에선 15·26 대신:
# vectorstore    = Chroma(client=client, collection_name="langchain",       embedding_function=embeddings)
# question_store = Chroma(client=client, collection_name="chunk_questions", embedding_function=embeddings)


저장 완료: ['langchain', 'chunk_questions']


## 7. 생성 모델까지 포함한 정식 평가 (API 살아있을 때 실행)

`my_answer()`를 팀 공용 반환 형식(`{"answer": str, "candidates": [...]}`)에 맞춰서 완성하고,
`run_and_save()`로 채점 + `experiments/results/{EXPERIMENT_NAME}.json` 저장까지 한 번에 처리한다.
API가 막혀있으면 이 셀은 건너뛰고 위 6번까지만으로도 청킹/임베딩 품질은 확인 가능하다.

In [36]:
from pipeline.evaluate import run_and_save

EXPERIMENT_NAME = "siyeon_exp01"  

NOTES = (
    "고정 길이 청킹 대신 LG 매뉴얼의 실제 목차 구조를 그대로 활용한 청킹. "
    "PDF 북마크(TOC) 2·3단계 소제목을 청크 경계로 쓰고(관리하기 > 청소하기 > 필터 청소하기 등), "
    "사용법 챕터는 목차에 없는 4단계 소제목(아이스 쿨파워 냉방 등)까지 감지, "
    "고장신고 섹션은 PyMuPDF 표 추출로 '증상 1개 = 청크 1개'(원인·해결책 전부 포함). "
    "페이지 번호/러닝헤더 제거. 임베딩은 한국어 구어체 질문과 격식체 매뉴얼 문장 매칭을 위해 "
    "다국어 특화 모델(bge-m3). 검색은 등록된 가전(doc_id) 메타데이터 필터 + LG 공통 에러코드(JSON, 1항목=1청크) 항상 포함."
)

STRATEGY = {
    "chunking":         "PDF 목차(TOC) 2·3단계 소제목 경계 청킹 + 고장신고 표는 증상 단위 (고정 길이 아님)",
    "chunking_reason":  (
        "LG 매뉴얼은 프레임메이커로 만들어져 목차가 PDF 북마크로 내장돼 있고, 10개 모델 전부 "
        "2·3단계 제목이 본문 줄과 글자 그대로 일치함 → 이 경계를 쓰면 '필터 청소하기'(청소 주기 표 포함), "
        "'냄새에 따른 제거 방법' 같은 질문 단위와 청크 단위가 일치함. 이전(1200자 고정 분할)에는 청소 주기 표가 "
        "두 청크로 갈리고 냄새 섹션이 다른 내용 뒤에 붙어 top-2/3이 엉뚱하게 나왔음. "
        "고장신고 표는 원인 1행=청크 1개로 하면 같은 증상 행이 top_k를 독식하고(와이파이 6행) 짧은 행이 전체 청크의 "
        "절반을 차지해 다른 섹션까지 밀어내서, 증상 1개=청크 1개로 합침(471→211개)"
    ),
    "embedding":        "BAAI/bge-m3 (로컬, sentence-transformers)",
    "embedding_reason": (
        "OpenAI 임베딩은 영어 중심이라 한국어 구어체 질문('에어컨이 안 시원해요')과 "
        "격식체 매뉴얼 문장 사이 매칭이 약함. bge-m3는 다국어 검색 특화 모델이고 "
        "로컬 무료라 API 키 상태와 무관하게 재현 가능"
    ),
    "db":               "ChromaDB 인메모리 (LangChain Document + langchain_chroma)",
    "db_reason":        (
        "등록된 가전(모델)별로 검색 범위를 좁혀야 해서 메타데이터 필터(doc_id)가 "
        "필요함. numpy 코사인으로도 되지만 LangChain Document/retriever 형태가 "
        "코드가 짧고, 나중에 persist_directory로 디스크 저장까지 그대로 이어짐"
    ),
    "retrieval":        (
        "[RDB 라우터] 에러코드는 SQLite 조회로 1등 고정, 기능 유무는 모델별 기능 표로 판정 → 나머지: "
        "doc_id(등록 모델) 필터 → 벡터(본문) + 벡터(청크별 사전 생성 질문) + 질문에 코드가 있을 때만 BM25(kiwi) "
        "→ RRF 융합 후보 8개 → bge-reranker-v2-m3 리랭킹 → top_k=3"
    ),
    "retrieval_reason": (
        "자체 테스트셋 63케이스로 실측: 벡터만 top1 83%/hit@3 92% → 리랭커+질문확장 88%/100%. "
        "BM25를 항상 섞으면 66%로 악화돼서 코드 질문에만 한정. 리랭커 후보 12→8, 길이 512→384가 더 빠르고 정확"
    ),
}

In [40]:
# 리랭커 distance(= 1 - 관련확률)가 이 값 이상인 후보는 LLM 컨텍스트에서 뺀다 (관련 확률 10% 미만).
# 팀 채점은 등록 가전 없이 전체 검색이라 제품군이 다른 후보(세탁기 질문에 에어컨 소음 행 등)가 들어오고,
# LLM이 그걸 근거로 답해버리는 걸 실측했음. 컷을 넣으면 "문서에 없다"고 답하게 된다.
# candidates(채점용 반환값)에는 컷과 무관하게 검색 결과를 전부 남긴다. None이면 컷 없음.
CONTEXT_CUTOFF = 0.9


def make_answer_fn(doc_id: str | list[str] | None = None, top_k: int = 3, search_fn=None,
                   context_cutoff: float | None = CONTEXT_CUTOFF):
    """doc_id를 고정한 my_answer(query)를 만들어 준다. search_fn 기본은 baseline search, 고도화는 search_v2/v3.

    run_and_save()는 my_answer(query) 한 개 인자만 넘기기 때문에, 모델 필터를
    쓰려면 이렇게 클로저로 doc_id를 미리 묶어야 한다.
      - 팀 공용 채점  : make_answer_fn(None)          -> 전체 문서 검색
      - 내 모델 테스트: make_answer_fn(TARGET_MODEL)  -> 등록된 가전으로만 검색
    context_cutoff: 리랭커 기반 search_v2/v3에서만 의미 있음 (baseline search의 L2 거리는 스케일이 달라 0.9가 너무 느슨함).
    """
    from openai import OpenAI
    import os

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY", ""))

    retrieve = search_fn or search

    def my_answer(query: str) -> dict:
        """팀 공용 평가 형식: {"answer": str, "candidates": [{"id","text","distance"}]}"""
        hits = retrieve(query, doc_id=doc_id, top_k=top_k)
        # 컷: 관련성이 낮은 후보는 컨텍스트에서 제외. 전부 걸러지면 LLM에 "관련 문서 없음"을 명시해서 지어내지 않게 한다
        used = [(c, d) for c, d in hits if context_cutoff is None or d < context_cutoff]
        context = "\n\n".join(f"[{c['section']} > {c['subsection']}]\n{c['body']}" for c, _ in used) \
            or "(검색된 문서 중 이 질문과 관련 있는 내용이 없습니다. 문서에 없다고 안내하세요.)"

        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"[검색된 문서]\n{context}\n\n[질문]\n{query}"},
            ],
        )
        return {
            "answer": resp.choices[0].message.content,
            "candidates": [
                {"id": c["id"], "text": c["text"], "distance": round(distance, 4)}
                for c, distance in hits
            ],
        }

    return my_answer


# 내 모델(TARGET_MODEL)로 한 질문만 빠르게 확인 (고도화 검색):
make_answer_fn(TARGET_MODEL, search_fn=search_v3)("취침 모드 어떻게 켜요?")   # 최종 라우터. search_v2/search도 가능

# 팀 공용 정식 채점 (API 살아있을 때). COMMON_QUESTIONS는 여러 제품군이 섞여 있어 전체 검색으로:
# run_and_save()는 상대경로 "experiments/results/"에 저장하는데 노트북 cwd가 experiments/라서
# experiments/experiments/results/ 에 떨어진다 → 프로젝트 루트로 잠시 이동해서 실행 (공용 evaluate.py는 수정 금지)
import os
_cwd = os.getcwd(); os.chdir(Path.cwd().resolve().parents[1])
try:
    report = run_and_save(EXPERIMENT_NAME, NOTES, STRATEGY, make_answer_fn(None, search_fn=search_v3))
finally:
    os.chdir(_cwd)


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.64it/s]



  실험명  : siyeon_exp01
  메모    : 고정 길이 청킹 대신 LG 매뉴얼의 실제 목차 구조를 그대로 활용한 청킹. PDF 북마크(TOC) 2·3단계 소제목을 청크 경계로 쓰고(관리하기 > 청소하기 > 필터 청소하기 등), 사용법 챕터는 목차에 없는 4단계 소제목(아이스 쿨파워 냉방 등)까지 감지, 고장신고 섹션은 PyMuPDF 표 추출로 '증상 1개 = 청크 1개'(원인·해결책 전부 포함). 페이지 번호/러닝헤더 제거. 임베딩은 한국어 구어체 질문과 격식체 매뉴얼 문장 매칭을 위해 다국어 특화 모델(bge-m3). 검색은 등록된 가전(doc_id) 메타데이터 필터 + LG 공통 에러코드(JSON, 1항목=1청크) 항상 포함.



Batches: 100%|██████████| 1/1 [00:00<00:00, 10.30it/s]


[O] 에러코드 단순 조회 (LG 세탁기, 원문 그대로 유지): 에어컨 UE 오류가 뭐야?
    elapsed=5.1s | avg_dist=0.9925


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.99it/s]


...

[O] 안전 관련 (브랜드 미언급): 에어컨에서 타는 냄새 나는데 계속 써도 돼?
    elapsed=4.8s | avg_dist=0.9166


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.19it/s]


[O] 모호한 질문 (브랜드 미언급): 냉장고에서 이상한 소리 나는데 고장인가요?
    elapsed=5.4s | avg_dist=0.3908


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.34it/s]


[O] 증상 기반 (브랜드 미언급): 냉장고 문이 잘 안 닫히는데 어떻게 고쳐?
    elapsed=5.7s | avg_dist=0.9953

  키워드 히트율  : 91%
  평균 거리      : 0.7204
  평균 응답 시간 : 5.3s

[저장] experiments\results\siyeon_exp01.json
